# Limpeza e leitura dos Dados


In [4]:
#Importação das bibliotecas
import pandas as pandas
import numpy as numpy
import warnings

warnings.filterwarnings("ignore")
print("Bibliotecas importadas") 

Bibliotecas importadas


In [5]:
# Carregamento dos dados do dataset original
caminho_dataset_original = 'RECLAMEAQUI_BIGLOJAS.csv'
df_original = pandas.read_csv(caminho_dataset_original)

print(f"Dataset carregado com sucesso - TOTAL DE REGISTROS:  {df_original.shape[0]} - TOTAL DE COLUNAS : {df_original.shape[1]}")

display(df_original.head(3))

Dataset carregado com sucesso - TOTAL DE REGISTROS:  1000 - TOTAL DE COLUNAS : 16


,ID,TEMA,LOCAL,TEMPO,CATEGORIA,STATUS,DESCRICAO,URL,ANO,MES,DIA,DIA_DO_ANO,SEMANA_DO_ANO,DIA_DA_SEMANA,TRIMETRES,CASOS
0,120300953,Carne estragada,Guarulhos - SP,2021-01-03,BIG - Lojas Físicas<->Produtos estragados<->Hi...,Respondida,No sábado dia 27/02/21 comprei 11espetos que e...,https://www.reclameaqui.com.br//big-hipermerca...,2021,1,3,3,53,6,1,1
1,121856411,Produto com defeito,São Paulo - SP,2021-01-04,BIG - Lojas Físicas<->Estorno do valor pago<->...,Respondida,No dia 19/03/2021 fui walmart de São Judas com...,https://www.reclameaqui.com.br//big-hipermerca...,2021,1,4,4,1,0,1,4
2,121896031,Pedido incompleto,São Paulo - SP,2021-01-04,Problema com entrega de produto-compras<->BIG ...,Não resolvido,"Fiz um pedido pelo ifood, paguei e na hora da ...",https://www.reclameaqui.com.br//big-hipermerca...,2021,1,4,4,1,0,1,4


In [6]:
# Tratamento de conversão do tipo datas 
print("Dados originais da coluna TEMPO: ", df_original['TEMPO'].dtype)

#converte a string de data para o formato datetime da biblioteca pandas
df_original['TEMPO'] = pandas.to_datetime(df_original['TEMPO'], errors='coerce')
print("Dados convertidos da coluna TEMPO: ", df_original['TEMPO'].dtype)

Dados originais da coluna TEMPO:  object
Dados convertidos da coluna TEMPO:  datetime64[ns]


In [8]:
# Tratamento de dados de separação de cidade e estado
# É criado duas novas colunas, uma para cidade e outra para estado e dividindo a string pelo " - "
if 'LOCAL' in df_original:
    df_original[['CIDADE', 'ESTADO']] = df_original['LOCAL'].str.split(' - ', expand=True)
    df_original['CIDADE'] = df_original['CIDADE'].str.strip()
    df_original['ESTADO'] = df_original['ESTADO'].str.strip()
display(df_original[['LOCAL', 'CIDADE', 'ESTADO']].head(3))

# Foi removido no str.strip() os espaços em brancos que podem atrapalhar no filtro

,LOCAL,CIDADE,ESTADO
0,Guarulhos - SP,Guarulhos,SP
1,São Paulo - SP,São Paulo,SP
2,São Paulo - SP,São Paulo,SP


In [ ]:
# Tratamento de padronização de texto categoricos e higenização dos dados
# Colocando temas e categorias em maisculo para evitar a mesma palara sejam consideradas diferentes 
colunas_texto = ['TEMA', 'CATEGORIA', 'STATUS']

for col in colunas_texto:
    df_original[col] = df_original[col].str.upper().str.strip().str.strip()

print("Textos padronizados: ")
print(df_original['STATUS'].unique())

#o for ele transforma em string e coloca tudo em maisculo e remove os espaços na ponta

Textos padronizados: 
['RESPONDIDA' 'NÃO RESOLVIDO' 'EM RÉPLICA' 'RESOLVIDO' 'NÃO RESPONDIDA']


In [12]:
# Tratamento de dados nulos e verificação
print("Contagem de nulos pré tratamento: ")
print(df_original.isnull().sum()[df_original.isnull().sum() > 0])

# Preenchimento de possiveis nulos nas cidades e estados caso algum no 'LOCAL' não tivesse hifen
df_original['CIDADE'].fillna('CIDADE NÃO INFORMADA', inplace=True)
df_original['ESTADO'].fillna('ESTADO NÃO INFORMADO', inplace=True)

# Preenchimento de descrição nula
df_original['DESCRICAO'].fillna('SEM DESCRIÇÃO', inplace=True)

print("\nTratamento dos dados nulos conluido")

Contagem de nulos pré tratamento: 
Series([], dtype: int64)

Tratamento dos dados nulos conluido


In [ ]:
# Tratamento e hierarquia de Categorias
# Os dados nessa coluna tem o nivel de estruturação de (Problema geral <-> Loja <-> Setor <-> Produto)
print("Separa da coluna CATEGORIA em niveis de hierarquia")

# O 'expand=True' transforma cada pedaço separado pelo '<->' em uma nova coluna
categorias_separadas = df_original['CATEGORIA'].str.split('<->', expand=True)
categorias_separadas.columns = ['CATEGORIA_NIVEL_1', 'CATEGORIA_NIVEL_2', 'CATEGORIA_NIVEL_3', 'CATEGORIA_NIVEL_4']
df_original = pandas.concat([df_original, categorias_separadas], axis=1)

colunas_niveis = ['CATEGORIA_NIVEL_1', 'CATEGORIA_NIVEL_2', 'CATEGORIA_NIVEL_3', 'CATEGORIA_NIVEL_4']
for col in colunas_niveis:
    df_original[col].fillna('NÃO ESPECIFICADO', inplace=True)
    df_original[col] = df_original[col].str.strip()

display(df_original[['CATEGORIA', 'CATEGORIA_NIVEL_1', 'CATEGORIA_NIVEL_2', 'CATEGORIA_NIVEL_3', 'CATEGORIA_NIVEL_4']].head(3))

Separa da coluna CATEGORIA em niveis de hierarquia


,CATEGORIA,CATEGORIA_NIVEL_1,CATEGORIA_NIVEL_2,CATEGORIA_NIVEL_3,CATEGORIA_NIVEL_4
0,BIG - LOJAS FÍSICAS<->PRODUTOS ESTRAGADOS<->HI...,BIG - LOJAS FÍSICAS,PRODUTOS ESTRAGADOS,HIPERMERCADOS,NÃO ESPECIFICADO
1,BIG - LOJAS FÍSICAS<->ESTORNO DO VALOR PAGO<->...,BIG - LOJAS FÍSICAS,ESTORNO DO VALOR PAGO,ELETRODOMÉSTICOS,MICRO-ONDAS
2,PROBLEMA COM ENTREGA DE PRODUTO-COMPRAS<->BIG ...,PROBLEMA COM ENTREGA DE PRODUTO-COMPRAS,BIG - LOJAS FÍSICAS,HIPERMERCADOS,NÃO ESPECIFICADO


In [19]:
# Geração do novo dataset agora com base tratada
# Remoção de coluna que não sera usada ou redundante
colunas_para_remover = ['LOCAL', 'URL', 'CATEGORIA'] # Não é preciso a URL na dashboard e a coluna LOCAL foi dividida em cidade e estado
df_tratado = df_original.drop(columns=colunas_para_remover, errors='ignore')

# Salvando o novo dataset com separação de ponto e virgula 
nome_arquivo_saida = 'BIGLOJAS_DADOS_TRATADOS.csv'
df_tratado.to_csv(nome_arquivo_saida, index=False, sep=';', encoding='utf-8-sig')

print(f"Dataset salvo com sucesso com base limpa como '{nome_arquivo_saida}'")

Dataset salvo com sucesso com base limpa como 'BIGLOJAS_DADOS_TRATADOS.csv'
